<a href="https://colab.research.google.com/github/i2mmmmm/train_project/blob/main/20230820_rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LayerNormalization, MultiHeadAttention, Flatten
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Activation
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping


In [3]:
from google.colab import drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
s30_train = pd.read_csv('/content/drive/My Drive/철도/s30_train.csv')
s40_train = pd.read_csv('/content/drive/My Drive/철도/s40_train.csv')
s50_train = pd.read_csv('/content/drive/My Drive/철도/s50_train.csv')
s70_train = pd.read_csv('/content/drive/My Drive/철도/s70_train.csv')
s100_train = pd.read_csv('/content/drive/My Drive/철도/s100_train.csv')
c30_train = pd.read_csv('/content/drive/My Drive/철도/c30_train.csv')
c40_train = pd.read_csv('/content/drive/My Drive/철도/c40_train.csv')
c50_train = pd.read_csv('/content/drive/My Drive/철도/c50_train.csv')
c70_train = pd.read_csv('/content/drive/My Drive/철도/c70_train.csv')
c100_train = pd.read_csv('/content/drive/My Drive/철도/c100_train.csv')

In [27]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)

data_s_X, data_s_y = prepare_data(c100_train)

N_TIMESTEPS = 10
N_FEATURES = 34

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)


# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# Encoder: 3개의 LSTM 블록을 추가
x = LSTM(64, return_sequences=True)(inputs)
x = Dropout(0.5)(x)
x = LSTM(128, return_sequences=True)(x)
x = Dropout(0.5)(x)
x = LSTM(256, return_sequences=False)(x)
x = Dropout(0.5)(x)

# Fully Connected Layer로 디코딩
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(4)(x)

model_rnn = Model(inputs=inputs, outputs=outputs)

model_rnn.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

# 모델 학습
model_rnn.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])

# 예측
predictions_s = model_rnn.predict(X_test_s)

# 예측 결과 출력
print("Predictions for data_s")
print(predictions_s[:5])

Epoch 1/50
157/157 [==============================] - 27s 128ms/step - loss: 6.1841e-04 - mae: 0.0177 - val_loss: 3.9197e-04 - val_mae: 0.0135
Epoch 2/50
157/157 [==============================] - 20s 131ms/step - loss: 2.3695e-04 - mae: 0.0116 - val_loss: 4.2417e-04 - val_mae: 0.0141
Epoch 3/50
157/157 [==============================] - 20s 127ms/step - loss: 1.9471e-04 - mae: 0.0105 - val_loss: 5.0460e-04 - val_mae: 0.0159
Epoch 4/50
157/157 [==============================] - 19s 122ms/step - loss: 1.6936e-04 - mae: 0.0098 - val_loss: 4.4996e-04 - val_mae: 0.0152
Epoch 5/50
157/157 [==============================] - 21s 136ms/step - loss: 1.5122e-04 - mae: 0.0092 - val_loss: 5.1790e-04 - val_mae: 0.0162
Epoch 6/50
157/157 [==============================] - 19s 123ms/step - loss: 1.4480e-04 - mae: 0.0090 - val_loss: 5.3181e-04 - val_mae: 0.0163
Epoch 7/50
157/157 [==============================] - 20s 124ms/step - loss: 1.3099e-04 - mae: 0.0085 - val_loss: 5.5553e-04 - val_mae: 0.0170

In [10]:
predictions_30s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_30s.to_csv('/content/drive/My Drive/철도/30s_20230820_rnn.csv', index=False)

In [12]:
predictions_40s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_40s.to_csv('/content/drive/My Drive/철도/40s_20230820_rnn.csv', index=False)

In [14]:
predictions_50s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_50s.to_csv('/content/drive/My Drive/철도/50s_20230820_rnn.csv', index=False)

In [16]:
predictions_70s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_70s.to_csv('/content/drive/My Drive/철도/70s_20230820_rnn.csv', index=False)

In [18]:
predictions_100s = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_100s.to_csv('/content/drive/My Drive/철도/100s_20230820_rnn.csv', index=False)

In [20]:
predictions_30c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_30c.to_csv('/content/drive/My Drive/철도/30c_20230820_rnn.csv', index=False)

In [22]:
predictions_40c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_40c.to_csv('/content/drive/My Drive/철도/40c_20230820_rnn.csv', index=False)

In [24]:
predictions_50c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_50c.to_csv('/content/drive/My Drive/철도/50c_20230820_rnn.csv', index=False)

In [26]:
predictions_70c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_70c.to_csv('/content/drive/My Drive/철도/70c_20230820_rnn.csv', index=False)

In [28]:
predictions_100c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1', 'YL_M1_B1_W2', 'YR_M1_B1_W2'])

# CSV 파일로 저장
predictions_100c.to_csv('/content/drive/My Drive/철도/100c_20230820_rnn.csv', index=False)

In [29]:
answer_sample = pd.read_csv('/content/drive/My Drive/철도/answer_sample.csv')
c30 = pd.read_csv('/content/drive/My Drive/철도/30c_20230820_rnn.csv')
c40 = pd.read_csv('/content/drive/My Drive/철도/40c_20230820_rnn.csv')
c50 = pd.read_csv('/content/drive/My Drive/철도/50c_20230820_rnn.csv')
c70 = pd.read_csv('/content/drive/My Drive/철도/70c_20230820_rnn.csv')
c100 = pd.read_csv('/content/drive/My Drive/철도/100c_20230820_rnn.csv')
s30 = pd.read_csv('/content/drive/My Drive/철도/30s_20230820_rnn.csv')
s40 = pd.read_csv('/content/drive/My Drive/철도/40s_20230820_rnn.csv')
s50 = pd.read_csv('/content/drive/My Drive/철도/50s_20230820_rnn.csv')
s70 = pd.read_csv('/content/drive/My Drive/철도/70s_20230820_rnn.csv')
s100 = pd.read_csv('/content/drive/My Drive/철도/100s_20230820_rnn.csv')

In [30]:
c30.columns = ['YL_M1_B1_W1_c30', 'YR_M1_B1_W1_c30', 'YL_M1_B1_W2_c30', 'YR_M1_B1_W2_c30']
c40.columns = ['YL_M1_B1_W1_c40', 'YR_M1_B1_W1_c40', 'YL_M1_B1_W2_c40', 'YR_M1_B1_W2_c40']
c50.columns = ['YL_M1_B1_W1_c50', 'YR_M1_B1_W1_c50', 'YL_M1_B1_W2_c50', 'YR_M1_B1_W2_c50']
c70.columns = ['YL_M1_B1_W1_c70', 'YR_M1_B1_W1_c70', 'YL_M1_B1_W2_c70', 'YR_M1_B1_W2_c70']
c100.columns = ['YL_M1_B1_W1_c100', 'YR_M1_B1_W1_c100', 'YL_M1_B1_W2_c100', 'YR_M1_B1_W2_c100']

s30.columns = ['YL_M1_B1_W1_s30', 'YR_M1_B1_W1_s30', 'YL_M1_B1_W2_s30', 'YR_M1_B1_W2_s30']
s40.columns = ['YL_M1_B1_W1_s40', 'YR_M1_B1_W1_s40', 'YL_M1_B1_W2_s40', 'YR_M1_B1_W2_s40']
s50.columns = ['YL_M1_B1_W1_s50', 'YR_M1_B1_W1_s50', 'YL_M1_B1_W2_s50', 'YR_M1_B1_W2_s50']
s70.columns = ['YL_M1_B1_W1_s70', 'YR_M1_B1_W1_s70', 'YL_M1_B1_W2_s70', 'YR_M1_B1_W2_s70']
s100.columns = ['YL_M1_B1_W1_s100', 'YR_M1_B1_W1_s100', 'YL_M1_B1_W2_s100', 'YR_M1_B1_W2_s100']

In [31]:
answer = pd.concat([answer_sample.Distance, s30,s40,s50,s70,s100,c30,c40,c50,c70,c100], axis=1)

In [32]:
answer

,Distance,YL_M1_B1_W1_s30,YR_M1_B1_W1_s30,YL_M1_B1_W2_s30,YR_M1_B1_W2_s30,YL_M1_B1_W1_s40,YR_M1_B1_W1_s40,YL_M1_B1_W2_s40,YR_M1_B1_W2_s40,YL_M1_B1_W1_s50,...,YL_M1_B1_W2_c50,YR_M1_B1_W2_c50,YL_M1_B1_W1_c70,YR_M1_B1_W1_c70,YL_M1_B1_W2_c70,YR_M1_B1_W2_c70,YL_M1_B1_W1_c100,YR_M1_B1_W1_c100,YL_M1_B1_W2_c100,YR_M1_B1_W2_c100
0,2500.25,0.024149,-0.007975,0.047614,-0.035504,-0.001529,0.007497,-0.004588,0.010016,-0.001662,...,-0.002053,0.007348,0.000970,0.008160,-0.003266,0.008275,0.001373,0.008127,0.002607,0.005046
1,2500.50,0.023815,-0.006700,0.053982,-0.041368,-0.005245,0.011193,-0.000173,0.005010,-0.005973,...,0.005578,0.000106,-0.000770,0.009619,0.005875,-0.000748,-0.000332,0.009933,0.009710,-0.001737
2,2500.75,0.020201,-0.003686,0.058479,-0.046686,-0.012620,0.018499,0.004483,-0.000362,-0.013828,...,0.015192,-0.008924,-0.006493,0.015267,0.016404,-0.011122,-0.005614,0.015685,0.017861,-0.009480
3,2501.00,0.014900,0.000286,0.058139,-0.048152,-0.019935,0.025669,0.006547,-0.003110,-0.021002,...,0.022291,-0.015456,-0.012388,0.021384,0.025544,-0.019983,-0.011447,0.022376,0.023914,-0.014926
4,2501.25,0.010534,0.003454,0.054968,-0.046716,-0.023161,0.029059,0.007175,-0.003866,-0.023846,...,0.027478,-0.020256,-0.014888,0.024082,0.032290,-0.026464,-0.014418,0.026018,0.027998,-0.018563
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1994,2998.75,0.008342,0.001639,0.032524,-0.026696,0.011624,-0.002459,0.043266,-0.030208,0.008494,...,0.031906,-0.021397,0.013775,-0.004643,0.036315,-0.025743,0.011895,-0.002112,0.031963,-0.019871
1995,2999.00,0.009797,0.000328,0.028429,-0.022550,0.012318,-0.003060,0.038354,-0.026019,0.009194,...,0.027979,-0.017881,0.013450,-0.004431,0.031907,-0.021919,0.013570,-0.003969,0.028842,-0.016832
1996,2999.25,0.009002,0.000627,0.020058,-0.015276,0.009686,-0.001535,0.028329,-0.018294,0.006975,...,0.020229,-0.011367,0.009625,-0.000841,0.022853,-0.014591,0.009861,-0.000246,0.021117,-0.010418
1997,2999.50,0.008045,0.001495,0.008182,-0.004563,0.009240,-0.002080,0.015030,-0.007361,0.006128,...,0.010395,-0.002780,0.007409,0.001510,0.011162,-0.004443,0.005889,0.003926,0.011834,-0.002498


In [34]:
answer.to_csv('/content/drive/My Drive/철도/answer20230820_rnn.csv', index=False)

In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 예측과 실제값 간의 MAE, MSE, RMSE, R-squared 계산
mae = mean_absolute_error(y_test_s, predictions_s)
mse = mean_squared_error(y_test_s, predictions_s)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_s, predictions_s)

# 결과 출력
print("Evaluation metrics for data_s:")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R-squared:", r2)


Evaluation metrics for data_s:
MAE: 0.010845244207797181
MSE: 0.0008102152430958973
RMSE: 0.028464280126079024
R-squared: 0.587409007435972


In [ ]:
# 예측 결과 DataFrame 생성
predictions_df = pd.DataFrame(predictions_s, columns=["Pred_YL_M1_B1_W1", "Pred_YR_M1_B1_W1", "Pred_YL_M1_B1_W2", "Pred_YR_M1_B1_W2"])

# CSV로 저장
predictions_df.to_csv("/Users/leeyoungdong/predictions100.csv", index=False)